# Plot POD data and process POD feeding buzz data

In [ ]:
from pathlib import Path

import pytz
from matplotlib import pyplot as plt
from pandas import (
    Timedelta,
)
from pandas.tseries import frequencies

from disclose.dataclass.detection_data import DetectionData
from disclose.utils.pod import (
    load_pod_folder,
    fit_gmm,
    gmm_feeding_buzz,
)

### Load data

In [ ]:
config = {
    "detection_file": Path(r"_static/POD_results.csv"),
    "filename_format": "%Y-%m-%d %H:%M:%S%z",
    "timebin_new": Timedelta(hours=1),
    "recording_file": Path(r"_static/POD_recording_periods.csv"),
}
data = DetectionData.from_dict(config)
data.change_tz(pytz.timezone("Etc/GMT-5"))
data.coordinates = (-49, 70)
data.add_dpm()

### Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(25, 6))
data.plot.histogram(
    ax=ax,
    annotator="cpod",
    label="sp_name",
    bin_size=frequencies.to_offset("1D"),
)
data.plot.set_ax(ax, x_ticks_res=frequencies.to_offset("1W-SUN"))
data.plot.feature(
    ax,
    effort=True,
    season=True,
    legend=False,
)
plt.show()

### Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(25, 6))
data.plot.heatmap(
    ax=ax,
    annotator="cpod",
    label="sp_name",
    bin_size=frequencies.to_offset("1W-SUN"),
)
data.plot.set_ax(ax, x_ticks_res=frequencies.to_offset("1W-SUN"))
data.plot.feature(
    ax,
    rise_set=True,
    effort=True,
    legend=False,
)
plt.show()

### Scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(25, 6))
data.plot.scatter(
    ax=ax,
    annotator="cpod",
    label="sp_name",
)
data.plot.set_ax(ax, x_ticks_res=frequencies.to_offset("1W-SUN"))
data.plot.feature(
    ax,
    rise_set=True,
    legend=False,
    effort=True,
)
plt.show()

### Process feeding buzz data

In [ ]:
fb_files = Path(
    r"L:\acoustock\DATASETS\COMMERSON\PROCESSED\DATA_ANALYSES\SITE B ILE HEUGH\Raw_FEEDINGBUZZES"
)
data_buzz = load_pod_folder(fb_files, ext="txt")
data_buzz, ici_log, gmm = fit_gmm(df=data_buzz, comp=3)
filtered_buzz = gmm_feeding_buzz(data_buzz, comp=3)
dpm_buzz = filtered_buzz.groupby(["start_datetime"])["fbm_count"].sum().reset_index()
new = data.df.merge(dpm_buzz, on=["start_datetime"], how="left")

In [ ]:
filtered_buzz["start_datetime"] = filtered_buzz["start_datetime"].dt.floor(
    "h"
)  # "h" permet de tronquer les dates du tableau à l'heure pour coller au format du tableau de DPM. Il est possible de travailler par 10min : "10min" ou par jour : "D".

sum_buzz = filtered_buzz.groupby(["start_datetime"])["fbm_count"].sum().reset_index()
sum_buzz["start_datetime"] = sum_buzz["start_datetime"].apply(
    lambda x: x.tz_convert(pytz.timezone("Etc/GMT-5"))
)

new = data.df.merge(sum_buzz, on=["start_datetime"], how="left")

new["FBR"] = new["fbm_count"] / new["dpm_count"]
print(
    f"{len(new[new['FBR'] > 1])} est le nombre de lignes où le FBR est supérieur à 1. Si ce nombre est différent de 0, il y a sans doute un problème dans l'analyse."
)
new[["FBR", "fbm_count"]] = new[["FBR", "fbm_count"]].fillna(0)

new_fb = new[new["fbm_count"] != 0]
fb_ap = DetectionData(df=new_fb, config=data.config)
fb_ap.coordinates = data.coordinates
fb_ap.change_tz(pytz.timezone("Etc/GMT-5"))

### Heatmap of feeding buzz

In [ ]:
fig, ax = plt.subplots(figsize=(25, 6))
fb_ap.plot.heatmap(
    ax=ax, annotator="cpod", label="sp_name", bin_size=frequencies.to_offset("1W-SUN")
)
fb_ap.plot.set_ax(ax, x_ticks_res=frequencies.to_offset("1W-SUN"))
fb_ap.plot.feature(
    ax,
    effort=True,
    season=False,
    legend=False,
    rise_set=True,
)
plt.show()